# День 3 — Baseline модель

**Цель:** первая рабочая модель (TF-IDF + Logistic Regression) как точка отсчёта.

Скриптовый аналог: [`train_baseline.py`](../train_baseline.py).

In [1]:
import sys, os
# Добавляем корень проекта в путь, чтобы импортировать модули (config, preprocess, ...)
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from config import MODELS_DIR, ID2LABEL
from train_baseline import get_split

## 1. Стратифицированный split

`get_split()` сохраняет индексы train/test в `reports/split_indices.json`, чтобы дни 4–5 сравнивались на том же test set.

In [3]:
df, train_idx, test_idx = get_split()
y_train = df.loc[train_idx, 'label'].to_numpy()
y_test = df.loc[test_idx, 'label'].to_numpy()
print('train:', len(train_idx), 'test:', len(test_idx))

train: 4257 test: 1065


## 2. TF-IDF (1,2) + Logistic Regression

In [4]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(df.loc[train_idx, 'text_clean'])
X_test = vectorizer.transform(df.loc[test_idx, 'text_clean'])

model = LogisticRegression(max_iter=200, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


## 3. Метрики

In [5]:
names = [ID2LABEL[i] for i in sorted(ID2LABEL)]
print(classification_report(y_test, y_pred, target_names=names, digits=4))
baseline_f1 = f1_score(y_test, y_pred, average='macro')
print(f'Baseline macro F1: {baseline_f1:.4f}')

              precision    recall  f1-score   support

    negative     0.6000    0.1780    0.2745       118
     neutral     0.7561    0.9201    0.8301       576
    positive     0.7356    0.6523    0.6914       371

    accuracy                         0.7446      1065
   macro avg     0.6972    0.5835    0.5987      1065
weighted avg     0.7316    0.7446    0.7202      1065

Baseline macro F1: 0.5987


## 4. Сохранение модели

In [6]:
joblib.dump(model, MODELS_DIR / 'baseline_model.pkl')
joblib.dump(vectorizer, MODELS_DIR / 'baseline_vectorizer.pkl')
print('Baseline сохранён.')

Baseline сохранён.
